In [20]:
import pandas as pd
import matplotlib.pyplot as plt

def ler_tabela(nome):
    df = pd.read_parquet(f"../data/processed/{nome}.parquet") 
    print(nome, df.shape)
    return df


pessoas = ler_tabela("pessoas")
sinistros = ler_tabela("sinistros")
veiculos = ler_tabela("veiculos")



pessoas (23211, 31)
sinistros (789122, 50)
veiculos (1015991, 12)


In [21]:
print(pessoas.info())

<class 'pandas.DataFrame'>
RangeIndex: 23211 entries, 0 to 23210
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   id_sinistro               23211 non-null  int64  
 1   id_pessoa                 23211 non-null  int64  
 2   id_veiculo                17390 non-null  float64
 3   cod_ibge                  23211 non-null  int64  
 4   municipio                 23211 non-null  str    
 5   regiao_administrativa     23211 non-null  str    
 6   tipo_via                  23211 non-null  str    
 7   tipo_veiculo_vitima       17901 non-null  str    
 8   tipo_de_vitima            23211 non-null  str    
 9   modo_transporte_vitima    23211 non-null  str    
 10  sexo                      23211 non-null  str    
 11  idade                     22153 non-null  float64
 12  gravidade_lesao           23211 non-null  str    
 13  faixa_etaria_demografica  23211 non-null  str    
 14  faixa_etaria_lega

In [22]:
print(pessoas.isna().mean())
print(pessoas.isna().sum())


id_sinistro                 0.000000
id_pessoa                   0.000000
id_veiculo                  0.250786
cod_ibge                    0.000000
municipio                   0.000000
regiao_administrativa       0.000000
tipo_via                    0.000000
tipo_veiculo_vitima         0.228771
tipo_de_vitima              0.000000
modo_transporte_vitima      0.000000
sexo                        0.000000
idade                       0.045582
gravidade_lesao             0.000000
faixa_etaria_demografica    0.000000
faixa_etaria_legal          0.000000
profissao                   0.636336
grau_de_instrucao           1.000000
nacionalidade               1.000000
data_sinistro               0.000000
ano_sinistro                0.000000
mes_sinistro                0.000000
dia_sinistro                0.000000
ano_mes_sinistro            0.000000
data_obito                  0.000000
ano_obito                   0.000000
mes_obito                   0.000000
dia_obito                   0.000000
a

tabelas com dados nulos
- id_veiculo = 0.25 | 5821
- tipo_veiculo_vitima = 0.22 | 5310 
- idade = 0.04 | 1058
- profissao = 0.63 | 14770
- grau_de_instrução = 1.00 | 23211
- nacionalidade = 1.00 | 23211


In [23]:
valores = pessoas.select_dtypes(include="number").agg(["min", "max", "mean", "median"])
print(valores)

         id_sinistro     id_pessoa    id_veiculo      cod_ibge       idade  \
min     2.456227e+06  5.000000e+00  3.300000e+01  3.500105e+06    0.000000   
max     2.845098e+06  2.241871e+06  3.291987e+06  3.557303e+06  102.000000   
mean    2.519887e+06  4.947594e+04  1.227387e+06  3.532503e+06   42.138672   
median  2.491011e+06  4.035900e+04  8.384050e+04  3.534401e+06   40.000000   

        ano_sinistro  mes_sinistro  dia_sinistro    ano_obito  mes_obito  \
min      2022.000000      1.000000      1.000000  2022.000000   1.000000   
max      2025.000000     12.000000     31.000000  2025.000000  12.000000   
mean     2023.557451      6.692258     15.680022  2023.560682   6.703546   
median   2024.000000      7.000000     16.000000  2024.000000   7.000000   

        dia_obito  tempo_sinistro_obito  
min       1.00000              0.000000  
max      31.00000             30.000000  
mean     15.72862              1.571712  
median   16.00000              0.000000  


mortes mensais sinistro/obito de onde vem os 0.11 de diferença?

mediana de idade de obitos = 40

ano com mais mortes - 2024
mes com mais mortes - agosto
dia com mais mortes - 16

tempo sinistro com 1.57 de media e 0 mediana, acredito que seja por falta de cadastro

In [ ]:
pessoas["data_sinistro"] = pd.to_datetime(pessoas["data_sinistro"], format="%d/%m/%Y")
pessoas["data_obito"] = pd.to_datetime(pessoas["data_obito"], format="%d/%m/%Y")



   id_sinistro  id_pessoa  id_veiculo  cod_ibge        municipio       regiao_administrativa  \
0      2479624      44067   2298533.0   3516408  FRANCO DA ROCHA  METROPOLITANA DE SÃO PAULO   

              tipo_via tipo_veiculo_vitima tipo_de_vitima modo_transporte_vitima       sexo  \
0  ESTRADAS E RODOVIAS           AUTOMOVEL       CONDUTOR              AUTOMOVEL  MASCULINO   

   idade gravidade_lesao faixa_etaria_demografica faixa_etaria_legal             profissao  \
0   42.0           FATAL                  40 a 44              40-44  AUXILIAR DE MECANICO   

  grau_de_instrucao nacionalidade data_sinistro data_obito               local_obito local_via  \
0               NaN           NaN    2022-01-01 2022-01-01  ESTABELECIMENTO DE SAUDE   PUBLICO   

   tempo_sinistro_obito  
0                   0.0  


In [69]:
# investigando diferença de %0.11 na media de mortes pós sinistro 

filtro_tempo = pessoas["data_obito"] != pessoas["data_sinistro"]

print(filtro_tempo.value_counts().sort_index())

pessoas["diferenca_dias_obito"] = (pessoas["data_obito"] - pessoas["data_sinistro"]).dt.days
print(pessoas["diferenca_dias_obito"].value_counts().sort_index())
print(pessoas["diferenca_dias_obito"].agg(["mean", "median"]))

False    17733
True      5478
Name: count, dtype: int64
diferenca_dias_obito
0     17733
1      1640
2       506
3       464
4       368
5       278
6       258
7       210
8       212
9       160
10      145
11      132
12      150
13      105
14       96
15       80
16       74
17       55
18       42
19       66
20       52
21       52
22       50
23       40
24       36
25       44
26       31
27       37
28       33
29       24
30       38
Name: count, dtype: int64
mean      1.571712
median    0.000000
Name: diferenca_dias_obito, dtype: float64


In [ ]:
# ajustando data, diminuindo colunas de 8 para 2

pessoas = pessoas.drop(
    columns=["ano_sinistro", "mes_sinistro", "dia_sinistro",
        "ano_mes_sinistro", "ano_obito", "mes_obito", "dia_obito", "ano_mes_obito"
])

print(pessoas.columns)

In [ ]:
# verificando congruência para idade maxima

pd.set_option({"display.width": 100, "display.max_columns":None})
analise_b = pessoas.loc[pessoas["idade"] == pessoas["idade"].max()]

print(analise_b)